In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Data & Pemisahan Variabel
df = pd.read_csv('wbc.csv')

# Eliminasi variabel tidak relevan
X = df.drop(columns=['id', 'Unnamed: 32', 'diagnosis'], errors='ignore')
y = df['diagnosis']

# 2. Encoding Kolom "diagnosis" (M = 1, B = 0)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Pembagian data latih dan uji (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 3, 4, 5, 6. Pembuatan Pipeline (Standardisasi + SelectKBest + Logistic Regression)
k_fitur = 10
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=k_fitur)),
    ('classifier', LogisticRegression(random_state=42))
])

# Pelatihan dan Evaluasi Model
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Akurasi Pengujian: {accuracy_score(y_test, y_pred):.4f}\n")
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# 7. Identifikasi Fitur Terbaik
mask = pipeline.named_steps['selector'].get_support()
fitur_terbaik = X.columns[mask].tolist()

print(f"Jumlah Fitur Terbaik: {len(fitur_terbaik)}")
print("Daftar Fitur Terbaik:", fitur_terbaik)

Akurasi Pengujian: 0.9561

Laporan Klasifikasi:
               precision    recall  f1-score   support

           B       0.95      0.99      0.97        72
           M       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Jumlah Fitur Terbaik: 10
Daftar Fitur Terbaik: ['radius_mean', 'perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean', 'radius_worst', 'perimeter_worst', 'area_worst', 'concavity_worst', 'concave points_worst']


Berdasarkan hasil analisis seleksi fitur, jumlah fitur terbaik yang paling optimal untuk digunakan adalah 10 fitur, yang terdiri dari concave points_worst, perimeter_worst, concave points_mean, radius_worst, perimeter_mean, area_worst, radius_mean, area_mean, concavity_mean, dan concavity_worst. Pemilihan sepuluh fitur utama ini berfokus pada variabel dimensi fisik serta kelengkungan sel tumor, sehingga mampu menghasilkan akurasi prediksi yang tinggi dan efisien tanpa perlu menggunakan keseluruhan data awal.